# Argus — Model Training (LSTM)

**Part of the model-training stage.** Trains the primary LSTM sequence model, including a
documented decision on the LSTM's hidden-state design. The RandomForest baseline that used to
live in this notebook has moved to
[`04_random_forest_training.ipynb`](./04_random_forest_training.ipynb) (it now trains on the flat
per-frame dataset instead of window-mean-aggregated features — see that notebook), and the
Spearman/Kruskal-Wallis feature-correlation analysis that used to precede it here has moved to
[`02_dataset_creation_flat.ipynb`](./02_dataset_creation_flat.ipynb), which is a more natural home
for it now that it runs on per-frame rather than per-window data.

**Reads:** `dataset_processed/lstm_windows.csv`, written by
[`01_dataset_creation_lstm.ipynb`](./01_dataset_creation_lstm.ipynb) — run that first. Each row is
one sliding window (1-6s), already zero-pre-padded to a fixed `MAX_TIMESTEPS = 60` and flattened
into `t{timestep}_f{feature}` columns, so **loading no longer means reading thousands of `.npy`
files** — it's a single `pd.read_csv` plus a reshape back to `(MAX_TIMESTEPS, num_features)`,
and no padding step happens in this notebook anymore (the CSV already stores the padded shape).

**Writes:** `models/feature_scaler_<VERSION>.joblib`, `models/lstm_drowsiness_<VERSION>.keras`.

This notebook does not import `GeometricRatioFeatureLayer` or touch MediaPipe/video at all — it
only reads the already-extracted, already-padded feature rows. `MAX_TIMESTEPS` and the per-frame
feature layout are taken as fixed constants here rather than derived from the class; if either
ever changes, it changes in `01_dataset_creation_lstm.ipynb` first, and the constants below must
be updated to match.


## Setup

Mounts Drive and defines the paths/constants this notebook needs. Independent of notebook 1's in-memory state — only depends on `lstm_windows.csv`.


In [ ]:
from google.colab import drive
import os

# Mount Google Drive in the Colab environment
drive.mount('/content/drive')

# Define the base root path for the Argus project
project_folder = "/content/drive/MyDrive/Argus"

print(f"Google Drive successfully mounted! Base project directory: {project_folder}")


In [ ]:
import os
import pandas as pd
import numpy as np

# --- Argus project paths (must match 01_dataset_creation_lstm.ipynb) ---
models_folder = f"{project_folder}/models"
dataset_folder = f"{project_folder}/dataset"
processed_folder = f"{dataset_folder}/dataset_processed"
lstm_windows_csv_path = os.path.join(processed_folder, "lstm_windows.csv")

if not os.path.exists(lstm_windows_csv_path):
    raise FileNotFoundError(
        f"'{lstm_windows_csv_path}' not found. Run 01_dataset_creation_lstm.ipynb first — this "
        "notebook only reads the dataset it produces, it doesn't build it."
    )

df_lstm_windows = pd.read_csv(lstm_windows_csv_path)
print(f"Loaded windowed dataset: {len(df_lstm_windows)} window rows, "
      f"{df_lstm_windows['subject'].nunique()} subjects.")


In [ ]:
# --- Feature layout constants (must match 01_dataset_creation_lstm.ipynb's Pipeline
# Configuration Constants) ---
MAX_TIMESTEPS = 60  # fixed pad length every window was padded to at generation time -- brought
                     # down from 120 after a full extraction run OOM-crashed the Colab kernel;
                     # see 01_dataset_creation_lstm.ipynb's Pipeline Configuration Constants cell

META_COLS = ["subject", "level", "parent_video", "window_duration_sec", "n_real_frames", "dropped_frames_in_video"]
feature_cols = [c for c in df_lstm_windows.columns if c not in META_COLS]

# num_features derived from the actual CSV columns, not hardcoded -- GeometricRatioFeatureLayer
# .blendshape_names has 51 entries (not the documented 52, missing "_neutral"), so the real
# per-frame count is 58, not 59. Deriving it here means it can't silently drift from the data.
if len(feature_cols) % MAX_TIMESTEPS != 0:
    raise ValueError(
        f"{len(feature_cols)} feature columns isn't divisible by MAX_TIMESTEPS={MAX_TIMESTEPS}; "
        "did 01_dataset_creation_lstm.ipynb's MAX_TIMESTEPS change without updating this notebook?"
    )
num_features = len(feature_cols) // MAX_TIMESTEPS

print(f"MAX_TIMESTEPS={MAX_TIMESTEPS}, num_features={num_features} (derived from {len(feature_cols)} feature columns)")


In [ ]:
import joblib
import datetime

# --- Model Management Configuration ---
# Set to True if you want to ignore existing models and retrain everything
FORCE_RETRAIN = True

# Dynamic naming with versioning (timestamp based)
VERSION_STR = datetime.datetime.now().strftime("%Y%m%d_%H%M")

# Versioned artifact paths for this session
scaler_path = os.path.join(models_folder, f"feature_scaler_{VERSION_STR}.joblib")
lstm_model_path = os.path.join(models_folder, f"lstm_drowsiness_{VERSION_STR}.keras")

def get_latest_model(folder, prefix, extension):
    """Return the path of the most recently versioned model matching prefix/extension, or None."""
    if not os.path.exists(folder):
        return None
    files = [f for f in os.listdir(folder) if f.startswith(prefix) and f.endswith(extension)]
    if not files:
        return None
    return os.path.join(folder, sorted(files)[-1])

print(f"✅ Model management initialized. Current version: {VERSION_STR}. Force retrain: {FORCE_RETRAIN}")


# LSTM Model for Drowsiness Detection

LSTMs are well-suited for sequential data like our windowed feature streams, since they can
capture temporal dependencies (e.g. how quickly the eyes are closing) that a single frame or a
window-mean can't express.

This section covers:
1.  **Data Preparation for LSTM:** Reshaping `lstm_windows.csv` rows into `(samples, timesteps, features)`.
2.  **LSTM Model Definition:** Building the neural network architecture using TensorFlow/Keras.
3.  **LSTM Model Training:** Compiling and training the model.
4.  **LSTM Model Evaluation:** Assessing the performance of the LSTM model.


## Data Preparation for LSTM

Each row of `lstm_windows.csv` is already a flattened, zero-pre-padded `(MAX_TIMESTEPS,
num_features)` window (see `01_dataset_creation_lstm.ipynb`'s "From thousands of `.npy` files to
one padded CSV" note) — so preparing `X_lstm` here is a `pandas` slice + `numpy` reshape, not a
per-window disk read and pad. No padding decision is made in this notebook; it was already made,
identically for every window, at generation time.


In [ ]:
import numpy as np
import pandas as pd

# Reshape every row's flattened feature columns back into (MAX_TIMESTEPS, num_features).
# feature_cols is already ordered t000_f00 ... t{MAX_TIMESTEPS-1}_f{num_features-1} because
# that's the order 01_dataset_creation_lstm.ipynb wrote them in.
X_lstm = df_lstm_windows[feature_cols].to_numpy(dtype=np.float32).reshape(-1, MAX_TIMESTEPS, num_features)

# Labels: 'level' is already the 3-class label (1=Alert, 2=Low Vigilant, 3=Drowsy), 0-indexed
# for categorical cross-entropy.
y_lstm = (df_lstm_windows['level'].to_numpy(dtype=np.int32) - 1)
groups_lstm = df_lstm_windows['subject'].to_numpy()

print(f"X_lstm shape: {X_lstm.shape}")  # (samples, MAX_TIMESTEPS, num_features)
print(f"y_lstm shape: {y_lstm.shape}, classes: {np.unique(y_lstm)}")

# Display a sample padded sequence -- last 5 timesteps, not first, since pre-padding means the
# most recent (guaranteed-real) frames are at the end of the sequence, not the start.
print("\nSample X_lstm (first sample, last 5 timesteps, first 5 features):")
print(X_lstm[0, -5:, :5])
print("\nSample y_lstm (first 5 labels):")
print(y_lstm[:5])


## Splitting and Scaling Data for LSTM

Now, we'll split the prepared `X_lstm` and `y_lstm` into training and testing sets. It's crucial to apply `StandardScaler` to the feature dimension of our 3D data. Since `StandardScaler` expects 2D input, we will reshape our `X_lstm` temporarily for scaling, and then reshape it back to 3D. We will also stratify the split to maintain the distribution of drowsiness levels in both sets.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import joblib

# Split data into training and testing sets, grouped by subject, so overlapping windows from the
# same subject/clip can't straddle train/test.
groups_lstm_arr = np.array(groups_lstm)
gss_lstm = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_lstm, test_idx_lstm = next(gss_lstm.split(X_lstm, y_lstm, groups=groups_lstm_arr))
X_train_lstm, X_test_lstm = X_lstm[train_idx_lstm], X_lstm[test_idx_lstm]
y_train_lstm, y_test_lstm = y_lstm[train_idx_lstm], y_lstm[test_idx_lstm]

print(f"Train subjects: {len(set(groups_lstm_arr[train_idx_lstm]))}, "
      f"Test subjects: {len(set(groups_lstm_arr[test_idx_lstm]))} (disjoint by construction)")

print(f"X_train_lstm shape before scaling: {X_train_lstm.shape}")
print(f"X_test_lstm shape before scaling: {X_test_lstm.shape}")

# Initialize StandardScaler
scaler = StandardScaler()

# Reshape from 3D to 2D for scaling: (samples * timesteps, features)
X_train_reshaped = X_train_lstm.reshape(-1, num_features)
X_test_reshaped = X_test_lstm.reshape(-1, num_features)

# Fit only on real (non-padding) rows, so the padding zeros don't skew the mean/variance.
train_real_mask = ~np.all(X_train_reshaped == 0, axis=1)
scaler.fit(X_train_reshaped[train_real_mask])
X_train_scaled_reshaped = scaler.transform(X_train_reshaped)
X_test_scaled_reshaped = scaler.transform(X_test_reshaped)

# Reshape back to 3D: (samples, timesteps, features)
X_train_scaled = X_train_scaled_reshaped.reshape(X_train_lstm.shape)
X_test_scaled = X_test_scaled_reshaped.reshape(X_test_lstm.shape)

# Persist the scaler -- 03_deployment_export.ipynb loads the latest feature_scaler_*.joblib to
# build the deployed model's Normalization layer, so this is the only place it gets saved now
# that RandomForest (which used to also produce a same-named but differently-shaped scaler) has
# moved to its own notebook.
joblib.dump(scaler, scaler_path)
print(f"💾 Scaler saved: {scaler_path}")

print(f"\nSuccessfully split and scaled data for LSTM.")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"y_train_lstm shape: {y_train_lstm.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")
print(f"y_test_lstm shape: {y_test_lstm.shape}")


## LSTM Model Definition

We will now define the architecture for our LSTM model. The model will consist of an `Input` layer, one or more `LSTM` layers to capture temporal dependencies, `Dropout` layers for regularization, and a final `Dense` layer with `softmax` activation for multi-class classification (3 drowsiness levels).


## Design Decision: Persistent (`stateful=True`) Hidden State — Analyzed and Deferred

Before defining the LSTM below, it's worth being explicit about a design choice: this model
uses `stateful=False` (Keras' default), meaning every inference call resets the LSTM's hidden
and cell state to zero and recomputes the full recurrence over whatever window it's given. The
alternative — `stateful=True`, where the hidden state persists across separate calls instead of
resetting — was evaluated and **deferred**, not overlooked. The reasoning:

**What `stateful=True` actually changes.** A plain (non-stateful) LSTM already computes real
`h_t`/`c_t` gating *within* a single call — that's the whole LSTM mechanism, and it's exactly
what lets this model learn "eyes have been closing over the last few seconds" from a 60-frame
window. `stateful=True` only changes whether that hidden state is *carried over* as the starting
point for the *next separate call* instead of being reset to zero each time. It doesn't add
memory that isn't already there; it changes whose job it is to keep it.

**Why we're not building the training pipeline for it (yet):**

1. **Data pipeline mismatch.** Today's windows are independent, overlapping, single-labeled
   slices of a clip (stride 1s, 6 window durations, dropped on any face-lost frame) — deliberately
   NOT continuous per-clip sequences. A `stateful=True` regime needs continuous per-clip
   ordering, a fixed `batch_size`, and explicit `model.reset_states()` calls at clip boundaries,
   with the clip fed in sequential chunks (state carried within a clip, reset between clips) —
   the existing clip-level label can still be reused as the target for each chunk, so this
   wouldn't require new per-timestep labeling, but it is a materially different assembly step
   from what `01_dataset_creation_lstm.ipynb` currently does.
2. **A real safety-relevant footgun.** The deployed model would need an explicit reset policy
   for when persisted state gets zeroed (new trip? face lost for N seconds? never?). Getting
   this wrong means a driver's drowsy hidden state silently persisting into a different driver's
   session — for a DMS, that's a correctness/safety bug class, not just a performance quirk.
3. **No measured problem to solve.** The current model is two small LSTM layers (128 and 64
   units) over a 60×58 window — recomputing that from zero every ~100ms (10 FPS) is very unlikely
   to be a real bottleneck on a Raspberry Pi 5 CPU, but this hasn't been profiled in this repo.
   Taking on (1) and (2) to remove an unmeasured cost isn't a good trade without that
   measurement.

**Recommendation:** keep `stateful=False`. If Pi 5 inference latency is ever actually measured
to be a problem, that's the point to revisit this trade-off — with a number to justify it,
not just the theoretical O(1)-vs-O(max_timesteps) argument.

**A separate, more actionable finding from this same review:** the LSTM's train/test split
(and the RandomForest's, in `04_random_forest_training.ipynb`) previously used a plain
`train_test_split(..., stratify=...)` that mixed subjects across train and test, even though
subject-grouping information was already being computed and just not used. Since windows overlap
within a clip, this let near-duplicate windows leak across the split and inflated both models'
reported test accuracy. The split below is now group-aware (`GroupShuffleSplit`, grouped by
subject) — this is a real bug fix, independent of the `stateful` question, and should be re-run
before trusting any reported accuracy number in the titulación report.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, BatchNormalization, SpatialDropout1D, GaussianNoise
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
import numpy as np

input_shape = (X_train_scaled.shape[1], X_train_scaled.shape[2])
num_classes = len(np.unique(y_lstm))

# Advanced Regularized Architecture
model = Sequential([
    Input(shape=input_shape),
    # 1. Noise Injection: Forces the model to ignore micro-jitter in landmarks
    GaussianNoise(0.01),
    SpatialDropout1D(0.3),

    Dense(units=128, activation='relu', kernel_regularizer=l2(0.01)),
    BatchNormalization(),

    # 2. More selective LSTM layers
    LSTM(units=64, return_sequences=True, recurrent_dropout=0.4, kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.5),

    LSTM(units=32, recurrent_dropout=0.4, kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Dropout(0.5),

    Dense(units=32, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.5),
    Dense(units=num_classes, activation='softmax')
])

learning_rate=0.00001
# 3. Gradient Clipping: clipnorm prevents extreme updates that cause overfitting
optimizer = Adam(learning_rate=learning_rate, clipnorm=1.0)

model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## LSTM Model Training

Now we will train the LSTM model using our scaled training data (`X_train_scaled`, `y_train_lstm`) and validate it on the test set (`X_test_scaled`, `y_test_lstm`). We will use `EarlyStopping` to prevent overfitting and a `ModelCheckpoint` to save the best model weights.

**Why this LSTM training passes `class_weight`.** Without it, every training example
contributes equally to the loss regardless of its label — if one level has far fewer windows
than the others, the model can get away with being consistently wrong about that level and
still post a good-looking overall loss/accuracy, since the rare class barely moves the average.
`class_weight` multiplies each class's contribution to the loss inversely to how rare it is, so
misclassifying a rare class actually costs something during training instead of being nearly
free to ignore.

This matters more than usual here: per the "Data Collection Methodology" note in
`01_dataset_creation_lstm.ipynb`, the `Drowsy` class includes footage (originally "level 6, entering
microsleep") that had to be acted rather than self-recorded from genuine fatigue, which
plausibly makes it rarer in the dataset too — and `Drowsy` is the single most safety-critical
class this system exists to catch. The
RandomForest baseline above already uses `class_weight='balanced'`; the LSTM previously didn't.
Computed below from the actual training-set label distribution, not assumed.


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
import os
import numpy as np

print("🚀 Training LSTM...")

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
checkpoint = ModelCheckpoint(filepath=os.path.join(models_folder, 'best_lstm_model.keras'), monitor='val_accuracy', save_best_only=True)
# Swapped the previous custom AdaptiveLR callback (nudged LR up/down each epoch based on the
# *sign* of the val_accuracy delta — an unproven heuristic that mostly just oscillated near its
# floor) for the standard, well-tested ReduceLROnPlateau: halve the LR after `patience` epochs
# with no val_loss improvement, which is a much better-grounded signal than a single epoch's
# accuracy wobble.
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-7, verbose=1)

# Recalculate weights
weights = compute_class_weight('balanced', classes=np.unique(y_train_lstm), y=y_train_lstm)
weight_dict = dict(zip(np.unique(y_train_lstm).tolist(), weights))

history = model.fit(
    X_train_scaled, y_train_lstm,
    epochs=100,
    batch_size=64,
    validation_data=(X_test_scaled, y_test_lstm),
    callbacks=[early_stopping, checkpoint, reduce_lr],
    class_weight=weight_dict,
    verbose=1
)

model.save(lstm_model_path)

## LSTM Model Evaluation

After training, we will evaluate the performance of our LSTM model on the test set. We'll use a classification report to get detailed metrics like precision, recall, and F1-score for each drowsiness level, and plot the training history.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test_scaled, y_test_lstm, verbose=0)
print(f"\nLSTM Test Loss: {loss:.4f}")
print(f"LSTM Test Accuracy: {accuracy:.4f}")

# Make predictions on the test set
y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

# Display classification report
print("\nClassification Report (LSTM Model):")
CLASS_NAMES = ['Alert', 'Low Vigilant', 'Drowsy']
print(classification_report(y_test_lstm, y_pred, target_names=CLASS_NAMES))

# Plot Confusion Matrix
conf_matrix = confusion_matrix(y_test_lstm, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (LSTM Model)')
plt.show()

# Plot training history only if it was generated in the current session
# (i.e., if the model was trained, not loaded)
if 'history' in locals() or 'history' in globals():
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("\nSkipping training history plots: 'history' object not found (model was likely loaded, not trained in this session).")